# 02 · Compute Drive Times

對每個里質心，計算到最近的台南市立圖書館的時間。

支援兩種 method（由 cell 內的 `METHOD` 變數切換）：
- `haversine_30kmh` — 直線距離 ÷ 30 km/h（快、粗略）
- `osrm` — 真實道路路徑（慢、準）；會自動斷點續跑

輸出：`data/processed/village_to_nearest_library.csv`

In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
from tqdm.notebook import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from lib.geo import haversine_km, drive_minutes_from_km, safe_centroid_latlon

RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

VILLAGES_IN = RAW_DIR / "tainan_villages.geojson"
LIBRARIES_IN = RAW_DIR / "tainan_libraries.csv"
OUTPUT = PROC_DIR / "village_to_nearest_library.csv"

# 切換計算方式：先用 haversine 跑一遍，確認管線正常再切 osrm
# METHOD = "haversine_30kmh"   # or "osrm"
METHOD = "osrm"
SPEED_KMH = 30.0

In [2]:
villages = gpd.read_file(VILLAGES_IN)
libraries = pd.read_csv(LIBRARIES_IN)

print(f"Villages: {len(villages)}, Libraries: {len(libraries)}")

# 對每個里算質心，存成新欄位
centroids = villages.geometry.apply(lambda g: safe_centroid_latlon(g, source_crs="EPSG:4326"))
villages["centroid_lat"] = [c[0] for c in centroids]
villages["centroid_lon"] = [c[1] for c in centroids]
villages[["village_id", "village_name", "district", "centroid_lat", "centroid_lon"]].head()

Villages: 650, Libraries: 45


,village_id,village_name,district,centroid_lat,centroid_lon
0,67000190020,嘉南里,善化區,23.124799,120.328927
1,67000190019,嘉北里,善化區,23.144934,120.336007
2,67000010003,三仙里,新營區,23.311797,120.311178
3,67000010001,忠政里,新營區,23.315191,120.314403
4,67000010008,王公里,新營區,23.315487,120.318858


In [3]:
def nearest_library_haversine(lat: float, lon: float) -> dict:
    distances = libraries.apply(
        lambda r: haversine_km(lat, lon, r["lat"], r["lon"]),
        axis=1,
    )
    idx = distances.idxmin()
    return {
        "nearest_library": libraries.at[idx, "name"],
        "library_lat": libraries.at[idx, "lat"],
        "library_lon": libraries.at[idx, "lon"],
        "distance_km": float(distances.at[idx]),
    }


if METHOD == "haversine_30kmh":
    rows = []
    for _, v in tqdm(villages.iterrows(), total=len(villages), desc="haversine"):
        n = nearest_library_haversine(v["centroid_lat"], v["centroid_lon"])
        rows.append({
            "village_id": v["village_id"],
            "village_name": v["village_name"],
            "district": v["district"],
            "centroid_lat": v["centroid_lat"],
            "centroid_lon": v["centroid_lon"],
            **n,
            "drive_minutes": drive_minutes_from_km(n["distance_km"], speed_kmh=SPEED_KMH),
            "method": METHOD,
        })
    result = pd.DataFrame(rows)
    result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
    print(f"✅ Saved {len(result)} rows to {OUTPUT}")
    result.head()

In [4]:
from lib.osrm import OSRMClient, OSRMError, load_progress, save_progress_row

OSRM_PROGRESS = PROC_DIR / "osrm_progress.csv"
N_CANDIDATES = 3           # 先用 haversine 取最近 3 座，再 OSRM 算實際時間取最小
OSRM_REQUEST_DELAY_S = 0.2  # 對公開 server 友善


def compute_osrm_for_village(
    client: OSRMClient, v_lat: float, v_lon: float
) -> dict:
    # 先 haversine 排序取候選
    candidates = libraries.copy()
    candidates["hav_km"] = candidates.apply(
        lambda r: haversine_km(v_lat, v_lon, r["lat"], r["lon"]), axis=1
    )
    top = candidates.nsmallest(N_CANDIDATES, "hav_km")

    best: dict | None = None
    method = "osrm_driving"
    for _, lib in top.iterrows():
        try:
            minutes = client.route_duration_minutes(
                v_lat, v_lon, lib["lat"], lib["lon"]
            )
        except OSRMError as exc:
            # 整個候選都失敗會 fallback；單一失敗只是跳過此候選
            continue
        if best is None or minutes < best["drive_minutes"]:
            best = {
                "nearest_library": lib["name"],
                "library_lat": float(lib["lat"]),
                "library_lon": float(lib["lon"]),
                "distance_km": float(lib["hav_km"]),
                "drive_minutes": minutes,
            }

    if best is None:
        # 所有候選都失敗 → fallback 到 haversine，標記方法
        fallback = nearest_library_haversine(v_lat, v_lon)
        best = {
            **fallback,
            "drive_minutes": drive_minutes_from_km(fallback["distance_km"], SPEED_KMH),
        }
        method = "osrm_failed_fallback"

    best["method"] = method
    return best


if METHOD == "osrm":
    client = OSRMClient(request_delay_s=OSRM_REQUEST_DELAY_S)

    done = load_progress(OSRM_PROGRESS)
    print(f"Resume: already done {len(done)} / {len(villages)} villages")

    todo = villages[~villages["village_id"].isin(done.keys())].copy()
    print(f"To process: {len(todo)}")

    for _, v in tqdm(todo.iterrows(), total=len(todo), desc="OSRM"):
        r = compute_osrm_for_village(client, v["centroid_lat"], v["centroid_lon"])
        save_progress_row(
            OSRM_PROGRESS,
            village_id=v["village_id"],
            nearest_library=r["nearest_library"],
            library_lat=r["library_lat"],
            library_lon=r["library_lon"],
            distance_km=r["distance_km"],
            drive_minutes=r["drive_minutes"],
            method=r["method"],
        )

    # 合併 progress + village 中繼資料，輸出最終 csv
    done = load_progress(OSRM_PROGRESS)
    rows = []
    for _, v in villages.iterrows():
        d = done.get(v["village_id"])
        if d is None:
            continue
        rows.append({
            "village_id": v["village_id"],
            "village_name": v["village_name"],
            "district": v["district"],
            "centroid_lat": v["centroid_lat"],
            "centroid_lon": v["centroid_lon"],
            **{k: d[k] for k in ["nearest_library", "library_lat", "library_lon", "distance_km", "drive_minutes", "method"]},
        })
    result = pd.DataFrame(rows)
    result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
    print(f"✅ Saved {len(result)} rows to {OUTPUT}")
    n_fallback = (result["method"] == "osrm_failed_fallback").sum()
    if n_fallback:
        print(f"⚠️  {n_fallback} villages fell back to haversine due to OSRM errors")

Resume: already done 0 / 650 villages
To process: 650


OSRM:   0%|          | 0/650 [00:00<?, ?it/s]

✅ Saved 650 rows to /Users/linbangqi/draw-dis-to-lib/data/processed/village_to_nearest_library.csv


In [5]:
result = pd.read_csv(OUTPUT)
print(result["drive_minutes"].describe())
print("\nBy bin:")
import numpy as np
bins = [0, 5, 10, 15, 20, 30, np.inf]
result["bin"] = pd.cut(result["drive_minutes"], bins=bins, right=False)
print(result["bin"].value_counts().sort_index())

count    650.000000
mean       6.010026
std        4.291427
min        0.423333
25%        3.312500
50%        5.028333
75%        7.541250
max       51.410000
Name: drive_minutes, dtype: float64

By bin:
bin
[0.0, 5.0)      324
[5.0, 10.0)     241
[10.0, 15.0)     63
[15.0, 20.0)     14
[20.0, 30.0)      7
[30.0, inf)       1
Name: count, dtype: int64
